### Import Libraries

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, auc,
                             mean_squared_error, r2_score)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.svm import SVC
from collections import Counter
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from pandas.api.types import is_numeric_dtype

### Configure Settings

In [ ]:
pd.set_option('display.max_columns', None)
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

### Load datasets

In [ ]:
streamworks = pd.read_csv('../data/streamworks_user_data.csv')
print(f"streamworks: {streamworks.shape}")

print(f"Dataset shape: {streamworks.shape}\n")
print("Missing values:\n", streamworks.isnull().sum().sort_values(ascending=False), "\n")
print("Descriptive statistics:\n", streamworks.describe(include='all').T, "\n")
print("First 5 rows:\n", streamworks.head())


In [ ]:
def explore_df(df, name="Dataset"):
    print(f"--- {name} Overview ---")
    print(f"Shape: {df.shape}\n")
    
    print("Info:")
    df.info()
    
    print("\nMissing Values:")
    print(df.isnull().sum().sort_values(ascending=False))
    
    print("\nDescriptive Statistics:")
    display(df.describe(include='all').T)
    
    print("\nValue Counts:")
    display(df.value_counts())
    
    print("\nFirst 5 rows:")
    display(df.head())

explore_df(streamworks, "StreamWorks Data")


### Clean & Prepare the Data

#### Convert to datetime

In [ ]:
def try_parse_date(s):
    for fmt in ("%d-%m-%Y","%d-%m-%y","%Y-%m-%d"):
        try:
            return pd.to_datetime(s, format=fmt, dayfirst=True)
        except:
            continue
    return pd.to_datetime(s, dayfirst=True, errors='coerce')

streamworks['signup_date'] = streamworks['signup_date'].astype(str).apply(try_parse_date)
streamworks['last_active_date'] = streamworks['last_active_date'].astype(str).apply(try_parse_date)
print("Unique last_active_date values:", streamworks['last_active_date'].nunique())
display(streamworks[['signup_date','last_active_date']].head())

#### Create tenure_days and is_loyal

In [ ]:
streamworks['tenure_days'] = (streamworks['last_active_date'] - streamworks['signup_date']).dt.days

streamworks.loc[streamworks['tenure_days'] < 0, 'tenure_days'] = np.nan
streamworks['is_loyal'] = (streamworks['tenure_days'] > 180).astype(int)
 
print(streamworks[['tenure_days','is_loyal']].describe())
print("tenure_days nulls:", streamworks['tenure_days'].isnull().sum())

#### Missing-value strategy

In [ ]:
streamworks = streamworks.dropna(subset=['is_churned'])
streamworks = streamworks.dropna(subset=['user_id'])
 
streamworks['monthly_fee'] = streamworks['monthly_fee'].astype(float)
medians = streamworks.groupby('subscription_type')['monthly_fee'].median()
def impute_monthly_fee(row):
    if pd.isna(row['monthly_fee']):
        st = row['subscription_type']
        if pd.isna(st):
            return streamworks['monthly_fee'].median()
        return medians.get(st, streamworks['monthly_fee'].median())
    return row['monthly_fee']

streamworks['monthly_fee'] = streamworks.apply(impute_monthly_fee, axis=1)

num_impute_cols = ['average_watch_hours','age','mobile_app_usage_pct','complaints_raised','tenure_days']
for c in num_impute_cols:
    if c in streamworks.columns:
        streamworks[c] = streamworks[c].fillna(streamworks[c].median())

cat_cols = ['country', 'subscription_type', 'received_promotions', 'referred_by_friend', 'gender']

for c in cat_cols:
    if c in streamworks.columns:
        mode_val = streamworks[c].mode()[0] 
        streamworks[c] = streamworks[c].fillna(mode_val)

streamworks = streamworks.dropna(subset=['signup_date', 'last_active_date'])

streamworks['signup_month'] = streamworks['signup_date'].dt.to_period('M')

print(streamworks.isna().sum())

#### Encoding

In [ ]:
def preprocess_streamworks(df):
    # normalize
    if 'gender' in df.columns:
        df['gender'] = df['gender'].astype(str).str.strip().str.title().replace('Nan', 'Unknown')

    if 'country' in df.columns:
        df['country'] = df['country'].astype(str).str.strip().str.title()

    # map binaries
    binary_map = {'Yes': 1, 'No': 0}

    if 'received_promotions' in df.columns:
        df['received_promotions_bin'] = df['received_promotions'].map(binary_map).fillna(0).astype(int)

    if 'referred_by_friend' in df.columns:
        df['referred_by_friend_bin'] = df['referred_by_friend'].map(binary_map).fillna(0).astype(int)

    # derived features
    if {'average_watch_hours', 'monthly_fee'}.issubset(df.columns):
        df['watch_per_fee_ratio'] = df['average_watch_hours'] / (df['monthly_fee'] + 1e-6)

    if 'mobile_app_usage_pct' in df.columns:
        df['heavy_mobile_user'] = (df['mobile_app_usage_pct'] >= 75).astype(int)

    if 'average_watch_hours' in df.columns:
        median_watch = df['average_watch_hours'].median()
        df['low_watch_time'] = (df['average_watch_hours'] < median_watch).astype(int)

    if {'received_promotions_bin', 'low_watch_time'}.issubset(df.columns):
        df['promo_and_low_watch'] = (
            (df['received_promotions_bin'] == 1) &
            (df['low_watch_time'] == 1)
        ).astype(int)

    if 'age' in df.columns:
        df['age_group'] = pd.cut(
            df['age'],
            bins=[0, 25, 35, 45, 55, 100],
            labels=['16-25', '26-35', '36-45', '46-55', '56+']
        )

    # one-hot encoding
    ohe_columns = [c for c in ['subscription_type', 'country', 'gender'] if c in df.columns]
    if ohe_columns:
        ohe = OneHotEncoder(sparse_output=False,)
        ohe_array = ohe.fit_transform(df[ohe_columns])
        ohe_df = pd.DataFrame(ohe_array, columns=ohe.get_feature_names_out(ohe_columns), index=df.index)
        df = df.drop(columns=ohe_columns)
        df = pd.concat([df, ohe_df], axis=1)

    return df

streamworks = preprocess_streamworks(streamworks)

### Visualizations for insights

#### Boxplot average_watch_hours by churn

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=streamworks, x='is_churned', y='average_watch_hours')
plt.xticks([0,1], ['Retained','Churned'])
plt.ylabel('Average Watch Hours')
plt.xlabel('Is Churned')
plt.title('Watch hours distribution — churn vs retained')
plt.savefig('../reports/figures/02_average_watch_hours_by_churn.png')
plt.show()

#### Churn rate by subscription_type

In [ ]:
sub_cols = [c for c in streamworks.columns if c.startswith('subscription_type_')]

churn_by_sub = {}

for col in sub_cols:
    tier = col.replace('subscription_type_', '')
    churn_by_sub[tier] = streamworks[streamworks[col] == 1]['is_churned'].mean()

churn_by_sub = pd.Series(churn_by_sub).sort_values(ascending=False)
churn_by_sub.plot(kind='bar', figsize=(6,4))
plt.title('Churn rate by subscription tier')
plt.ylabel('Churn rate')
plt.savefig('../reports/figures/03_churn_rate_by_subscription_bar.png')
plt.show()

sub_cols = [c for c in streamworks.columns if c.startswith('subscription_type_')]
sub_counts = streamworks[sub_cols].sum()
labels = [c.replace('subscription_type_', '') for c in sub_cols]
colors = sns.color_palette('pastel', len(labels))

plt.figure(figsize=(6,6))
plt.pie(sub_counts, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors, explode=[0.05]*len(labels), shadow=True)
plt.title('Users by Subscription Tier')
plt.savefig('../reports/figures/04_churn_rate_by_subscription_pie.png')
plt.show()


#### Churn by age_group and mobile usage

In [ ]:
churn_age = streamworks.groupby(['age_group'], observed=False)['is_churned'].mean()
churn_age.plot(kind='bar', figsize=(6,4), color='mediumseagreen', edgecolor='black')
plt.title('Churn Rate by Age Group')
plt.ylabel('Churn Rate')
plt.xlabel('Age Group')
plt.savefig('../reports/figures/05_churn_rate_by_age_group.png')
plt.show()


#### Heavy Mobile User

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='heavy_mobile_user', hue='is_churned', data=streamworks, palette='Set2')
plt.title('Heavy Mobile User vs Churn')
plt.xlabel('Heavy Mobile User')
plt.ylabel('Count')
plt.xticks(ticks=[0,1], labels=['No', 'Yes'])
plt.legend(title='Churned', labels=['No', 'Yes'])
plt.savefig('../reports/figures/06_heavy_mobile_user_vs_churn_bar.png')
plt.show()

plt.figure(figsize=(6,6))
counts = streamworks['heavy_mobile_user'].value_counts()
labels = ['No', 'Yes']
colors = ['skyblue', 'orange']

plt.pie(counts, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors, explode=(0,0.1), shadow=True)
plt.title('Proportion of Heavy Mobile Users')
plt.savefig('../reports/figures/07_heavy_mobile_user_vs_churn_pie.png')
plt.show()


#### Churn rate by tenure_days

In [ ]:
bins = [0, 30, 90, 180, 365, streamworks['tenure_days'].max()]
labels = ['0-30','31-90','91-180','181-365','366+']
streamworks['tenure_bin'] = pd.cut(streamworks['tenure_days'], bins=bins, labels=labels, right=False)

churn_tenure = streamworks.groupby('tenure_bin', observed=False)['is_churned'].mean()

plt.figure(figsize=(8,5))
churn_tenure.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Churn Rate by Tenure Days')
plt.xlabel('Tenure Days Bin')
plt.ylabel('Churn Rate')
plt.ylim(0,1)
plt.savefig('../reports/figures/08_churn_rate_by_tenure_days.png')
plt.show()

#### Correlation analysis

In [ ]:
numeric_cols = streamworks.select_dtypes(include='number').columns.tolist()
corr_matrix = streamworks[numeric_cols].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', cbar=True)
plt.title('Correlation Matrix of Numeric Features')
plt.savefig('../reports/figures/01_correlation_heatmap.png')
plt.show()

#### Churn vs received promotions

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='received_promotions_bin', hue='is_churned', data=streamworks, palette='Set2')
plt.xticks([0,1], ['No','Yes'])
plt.title("Churn by Received Promotions")
plt.xlabel("Received Promotions")
plt.ylabel("Count")
plt.legend(title="Churned", labels=['No','Yes'])
plt.savefig('../reports/figures/09_churn_received_promotions.png')
plt.show()

#### Churn vs referred by friend

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='referred_by_friend_bin', hue='is_churned', data=streamworks, palette='Set2')
plt.xticks([0,1], ['No','Yes'])
plt.title("Churn by Referred by Friend")
plt.xlabel("Referred by Friend")
plt.ylabel("Count")
plt.legend(title="Churned", labels=['No','Yes'])
plt.savefig('../reports/figures/10_churn_referred_by_friend.png')
plt.show()

#### Scatter plot: tenure_days vs watch_per_fee_ratio colored by churn

In [ ]:
sns.lmplot(
    data=streamworks,
    x='tenure_days',
    y='watch_per_fee_ratio',
    hue='is_churned',
    palette={0: 'green', 1: 'red'},
    height=6,
    aspect=1.5,
    scatter_kws={'alpha':0.6}
)
plt.title('Tenure vs Watch per Fee Ratio by Churn')
plt.xlabel('Tenure (Days)')
plt.ylabel('Watch per Fee Ratio')
plt.savefig('../reports/figures/11_tenure_vs_watch_per_fee.png')
plt.show()

#### Bar plot: churn rate by received_promotions

In [ ]:
promo_churn = streamworks.groupby('received_promotions_bin')['is_churned'].mean()
promo_churn.plot(kind='bar', color='skyblue', edgecolor='black', figsize=(6,4))
plt.title('Churn Rate by Promotion Status')
plt.ylabel('Churn Rate')
plt.xlabel('Received Promotions')
plt.xticks([0,1], ['No', 'Yes'], rotation=0)
plt.savefig('../reports/figures/12_churn_rate_by_promotion.png')
plt.show()

#### Bar plot: churn rate by referred_by_friend

In [ ]:
ref_churn = streamworks.groupby('referred_by_friend_bin')['is_churned'].mean()
ref_churn.plot(kind='bar', color='mediumseagreen', edgecolor='black', figsize=(6,4))
plt.title('Churn Rate by Referral Status')
plt.ylabel('Churn Rate')
plt.xlabel('Referred by Friend')
plt.xticks([0,1], ['No', 'Yes'], rotation=0)
plt.savefig('../reports/figures/13_churn_rate_by_referral.png')
plt.show()

### Statistical analysis

#### Chi-square tests

In [ ]:
categorical_features = ['received_promotions_bin', 'referred_by_friend_bin']
gender_cols = [c for c in streamworks.columns if c.startswith('gender_')]

chi2_results = []

for col in categorical_features:
    contingency_table = pd.crosstab(streamworks[col], streamworks['is_churned'])
    chi2, p, dof, ex = chi2_contingency(contingency_table)
    chi2_results.append({
        'Feature': col,
        'Chi2': chi2,
        'p-value': p,
        'Significant': p < 0.05
    })

for col in gender_cols:
    contingency_table = pd.crosstab(streamworks[col], streamworks['is_churned'])
    chi2, p, dof, ex = chi2_contingency(contingency_table)
    chi2_results.append({
        'Feature': col,
        'Chi2': chi2,
        'p-value': p,
        'Significant': p < 0.05
    })

chi2_results_df = pd.DataFrame(chi2_results)
print("Chi-square test results for categorical variables (after preprocessing):")
display(chi2_results_df)

#### T-test

In [ ]:
watch_churned = streamworks.loc[streamworks['is_churned'] == 1, 'average_watch_hours']
watch_retained = streamworks.loc[streamworks['is_churned'] == 0, 'average_watch_hours']

t_stat, p_val = ttest_ind(watch_churned, watch_retained, equal_var=False)
print(f"\nT-test: Watch hours (Churned vs Retained)")
print(f"T-statistic: {t_stat:.3f}, p-value: {p_val:.4f}")

if p_val < 0.05:
    print("> There is a significant difference in watch hours between churned and retained users.")
else:
    print("> No significant difference in watch hours between churned and retained users.")
    
print(f"Mean watch hours - Churned: {watch_churned.mean():.2f}, Retained: {watch_retained.mean():.2f}")

### Predictive modelling

#### Logistic Regression

In [ ]:
exclude_cols = ['user_id', 'signup_date', 'last_active_date', 'is_churned']
X = streamworks.drop(columns=exclude_cols)
y = streamworks['is_churned']

categorical_cols = ['gender', 'subscription_type', 'country', 'received_promotions', 'referred_by_friend']
X = X.drop(columns=[c for c in categorical_cols if c in X.columns])

X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train_scaled, y_train)

y_pred = logreg.predict(X_test_scaled)
y_prob = logreg.predict_proba(X_test_scaled)[:,1]

print("=== Logistic Regression: Churn Prediction ===\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC AUC: {roc_auc:.3f}\n")

coef_df = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Coefficient': logreg.coef_[0]
}).sort_values(by='Coefficient', key=abs, ascending=False)

print("Top predictors of churn:")
display(coef_df.head(10))

##### ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr) 

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0,1], [0,1], color='navy', lw=2, linestyle='--', label='Random guess')
plt.fill_between(fpr, tpr, alpha=0.2, color='darkorange')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.savefig('../reports/figures/14_roc_curve.png')
plt.show()

#### Linear Regression

In [ ]:
target_lr = 'average_watch_hours'
y_lr = streamworks[target_lr]

X_lr = X.drop(columns=[target_lr])

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, y_lr, test_size=0.2, random_state=42
)

scaler_lr = StandardScaler()
X_train_lr_scaled = pd.DataFrame(scaler_lr.fit_transform(X_train_lr), columns=X_train_lr.columns)
X_test_lr_scaled = pd.DataFrame(scaler_lr.transform(X_test_lr), columns=X_test_lr.columns)

linreg = LinearRegression()
linreg.fit(X_train_lr_scaled, y_train_lr)

y_pred_lr = linreg.predict(X_test_lr_scaled)

rmse = mean_squared_error(y_test_lr, y_pred_lr) ** 0.5
r2 = r2_score(y_test_lr, y_pred_lr)

print("=== Linear Regression: Predict average_watch_hours ===\n")
print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")

coef_lr_df = pd.DataFrame({
    'Feature': X_train_lr_scaled.columns,
    'Coefficient': linreg.coef_
}).sort_values(by='Coefficient', key=abs, ascending=False)

print("Top features affecting average_watch_hours:")
display(coef_lr_df.head(10))

##### Residual plot

In [ ]:
plt.figure(figsize=(8,5))

residuals = y_test_lr - y_pred_lr

sns.scatterplot(x=y_pred_lr, y=residuals, alpha=0.6, color='teal', edgecolor=None)

plt.axhline(0, color='red', linestyle='--', linewidth=1.5, label='Zero Residual')

import statsmodels.api as sm
lowess = sm.nonparametric.lowess(residuals, y_pred_lr)
plt.plot(lowess[:,0], lowess[:,1], color='orange', linewidth=2, label='Trend (LOWESS)')

plt.xlabel('Predicted Average Watch Hours', fontsize=12)
plt.ylabel('Residuals', fontsize=12)
plt.title('Residuals vs Predicted: average_watch_hours', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/15_residual.png')
plt.show()


### Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10, 
    random_state=42, 
    class_weight='balanced'
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

print("=== Random Forest ===")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)
print(f"ROC AUC (Random Forest): {roc_auc_rf:.3f}")

### Segment churn by country or subscription type

In [ ]:
country_cols = [col for col in streamworks.columns if col.startswith("country_")]
streamworks["country"] = streamworks[country_cols].idxmax(axis=1).str.replace("country_", "")
churn_country = streamworks.groupby("country")["is_churned"].mean().sort_values(ascending=False)

churn_country.plot(kind='bar', figsize=(8,5), edgecolor='black')
plt.ylabel('Churn Rate')
plt.title('Churn Rate by Country')
plt.savefig('../reports/figures/16_churn_rate_by_country.png')
plt.show()


### Class Imbalance

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

rus = RandomUnderSampler(random_state=42)
X_train_ru, y_train_ru = rus.fit_resample(X_train, y_train)

print("Original:", Counter(y_train))
print("SMOTE:", Counter(y_train_sm))
print("Under-sampled:", Counter(y_train_ru))

### Model Tuning with GridSearchCV

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

param_grid = {
    'lr__C': [0.01, 0.1, 1, 10],
    'lr__solver': ['liblinear', 'lbfgs']
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best CV AUC:", grid.best_score_)


rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)
y_prob = rf.predict_proba(X_test)[:,1]

print("RF AUC:", roc_auc_score(y_test, y_prob))

### Alternative Models: SVM

In [ ]:
svm_model = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
svm_model.fit(X_train, y_train)
y_prob_svm = svm_model.predict_proba(X_test)[:, 1]
roc_auc_svm = roc_auc_score(y_test, y_prob_svm)
print(f"SVM ROC AUC: {roc_auc_svm:.3f}")

### Time Series / Trends with signup_date or last_active_date

In [ ]:
monthly_signups = streamworks.groupby('signup_month')['user_id'].count()
monthly_signups.plot(figsize=(10,4))
plt.title("Monthly Signups")
plt.ylabel("Number of Users")
plt.savefig('../reports/figures/17_monthly_signups.png')
plt.show()

monthly_churn = streamworks.groupby('signup_month')['is_churned'].mean()
monthly_churn.plot(figsize=(10,4), color='red')
plt.title("Monthly Churn Rate")
plt.ylabel("Churn Rate")
plt.savefig('../reports/figures/18_monthly_churn_rate.png')
plt.show()

### Business Questions

#### Do users who receive promotions churn less?

In [ ]:
ct = pd.crosstab(streamworks['received_promotions_bin'], streamworks['is_churned'])
print("Contingency table:\n", ct)

chi2, p, _, _ = chi2_contingency(ct)
print(f"Chi-square test: chi2={chi2:.2f}, p={p:.4f}")

churn_rates = streamworks.groupby('received_promotions_bin')['is_churned'].mean()
print("\nChurn rates:\n", churn_rates)

sns.barplot(x=churn_rates.index.map({0:'No',1:'Yes'}), y=churn_rates.values)
plt.ylabel("Churn Rate")
plt.title("Churn Rate by Promotions Received")
plt.show()

#### Does watch time impact churn likelihood?

In [ ]:
churned_watch = streamworks[streamworks['is_churned']==1]['average_watch_hours']
retained_watch = streamworks[streamworks['is_churned']==0]['average_watch_hours']

t_stat, p_val = ttest_ind(churned_watch, retained_watch, nan_policy='omit')
print(f"T-test: t={t_stat:.2f}, p={p_val:.4f}")

sns.boxplot(x='is_churned', y='average_watch_hours', data=streamworks)
plt.xticks([0,1], ['Retained','Churned'])
plt.ylabel("Average Watch Hours")
plt.title("Watch Time vs Churn")
plt.savefig('../reports/figures/19_watch_vs_churn.png')
plt.show()

#### Are mobile dominant users more likely to cancel?

In [ ]:
ct_mobile = pd.crosstab(streamworks['heavy_mobile_user'], streamworks['is_churned'])
chi2, p, _, _ = chi2_contingency(ct_mobile)
print("Contingency table:\n", ct_mobile)
print(f"Chi-square test: chi2={chi2:.2f}, p={p:.4f}")

mobile_rates = streamworks.groupby('heavy_mobile_user')['is_churned'].mean()
print("\nChurn rates:\n", mobile_rates)

sns.countplot(x='heavy_mobile_user', hue='is_churned', data=streamworks)
plt.xticks([0,1], ['Not mobile-dominant','Mobile-dominant'])
plt.title("Mobile Dominant Users vs Churn")
plt.savefig('../reports/figures/20_mobile_users_vs_churn.png')
plt.show()


#### Top 3 features influencing churn

In [ ]:
exclude_cols = ['is_churned','user_id']
X = streamworks.drop(columns=exclude_cols)
y = streamworks['is_churned']

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
datetime_cols = X.select_dtypes(include=['datetime64[ns]']).columns.tolist()

for col in datetime_cols:
    X[col] = (X[col] - X[col].min()).dt.days
    num_cols.append(col)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('rf', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42))
])

rf_pipeline.fit(X_train, y_train)

feature_names = num_cols + list(rf_pipeline.named_steps['preprocess'].named_transformers_['cat'].get_feature_names_out(cat_cols))
importances = pd.Series(rf_pipeline.named_steps['rf'].feature_importances_, index=feature_names).sort_values(ascending=False)

print("Top 3 features:\n", importances.head(3))

sns.barplot(x=importances.head(10).values, y=importances.head(10).index)
plt.title("Top 10 Feature Importances (Random Forest)")
plt.savefig('../reports/figures/21_top_10_feature.png')
plt.show()

#### Customer segments for retention prioritization

In [ ]:
subs = ['subscription_type_Basic', 'subscription_type_Premium', 'subscription_type_Standard']

streamworks['subscription_type'] = np.select(
    [streamworks[col]==1 for col in subs],
    ['Basic', 'Premium', 'Standard'],
    default='Unknown'
)

seg = streamworks.groupby(
    ['subscription_type','age_group'], observed=False
).agg(
    churn_rate=('is_churned','mean'),
    n_customers=('is_churned','size')
).reset_index()


seg = seg[seg['n_customers'] >= 30].sort_values('churn_rate', ascending=False)
print(seg.head(10))

plt.figure(figsize=(12,6))

ax = sns.barplot(
    data=seg,
    x='subscription_type',
    y='churn_rate',
    hue='age_group',
    palette='Set2',
    errorbar=None
)

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.2%}', 
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=10, color='black', xytext=(0,3),
                    textcoords='offset points')

plt.title("Churn Rate by Subscription Type and Age Group", fontsize=14, weight='bold')
plt.ylabel("Churn Rate")
plt.xlabel("Subscription Type")
plt.ylim(0, 1)
plt.legend(title='Age Group')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig('../reports/figures/22_churn_by_sub_age.png')
plt.show()


#### Factors affecting watch time / tenure (Linear Regression)

In [ ]:
target = 'average_watch_hours'
exclude_cols = ['user_id','signup_date','last_active_date','is_churned','average_watch_hours','tenure_bin','signup_month']

features = [c for c in streamworks.columns if c not in exclude_cols and is_numeric_dtype(streamworks[c])]
X_lr = streamworks[features].copy()
y_lr = streamworks[target].astype(float)

Xtr, Xte, ytr, yte = train_test_split(X_lr, y_lr, test_size=0.2, random_state=42)

num_cols_lr = Xtr.select_dtypes(include=['float64','int64']).columns.tolist()
scaler_lr = StandardScaler()
Xtr[num_cols_lr] = scaler_lr.fit_transform(Xtr[num_cols_lr])
Xte[num_cols_lr] = scaler_lr.transform(Xte[num_cols_lr])

lin = LinearRegression()
lin.fit(Xtr, ytr)

mse = mean_squared_error(yte, y_pred_lr)
rmse = np.sqrt(mse)
r2 = r2_score(yte, y_pred_lr)

print(f"Linear Regression for average_watch_hours: RMSE = {rmse:.3f}, R^2 = {r2:.3f}")

coeffs = pd.Series(lin.coef_, index=Xtr.columns).sort_values(key=abs, ascending=False)
display(coeffs.head(15))

residuals = yte - y_pred_lr
plt.figure(figsize=(8,5))
sns.scatterplot(x=y_pred_lr, y=residuals, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted average_watch_hours')
plt.ylabel('Residual (actual - predicted)')
plt.title('Residuals vs Predicted: average_watch_hours')
plt.savefig('../reports/figures/23_residuals_vs_predicted.png')
plt.show()